# Gatefall — ComfyUI on Google Colab (free GPU)

Free way to generate Gatefall character art without a local GPU. Runs
ComfyUI on Colab's free Tesla T4, downloads an anime checkpoint
(Illustrious or Pony Diffusion XL), and exposes the ComfyUI web UI via
a Cloudflare quick tunnel so you can use it in your browser.

**Before running:** `Runtime` menu -> `Change runtime type` -> select
`T4 GPU`, then `Save`. Run the cells below in order.

See `docs/art-direction.md` for the character prompts to paste in, and
`docs/comfyui-tutorial.md` for the full local-install version of this
guide.

**Colab free-tier limits:** sessions disconnect after inactivity and
have a rolling usage cap. Save any image you want to keep — closing
the tab or losing the session wipes the Colab disk.

## 1. Confirm the GPU is attached

In [ ]:
!nvidia-smi

## 2. Install ComfyUI

In [ ]:
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI
%cd /content/ComfyUI
!pip install -r requirements.txt -q

## 3. Download a checkpoint (Illustrious or Pony Diffusion XL)

Pick a checkpoint from Civitai and copy its **download URL**:

1. Go to civitai.com and search "Illustrious XL" or "Pony Diffusion
   XL", open the base checkpoint model page.
2. Right-click the **Download** button -> copy link address
   (looks like `https://civitai.com/api/download/models/XXXXXX`).
3. Paste it into `CHECKPOINT_URL` below.

Some Civitai models require being logged in to download. If the
download fails with an auth error, get an API key from
civitai.com -> account settings -> API Keys, and paste it into
`CIVITAI_TOKEN`. Leave it as an empty string if not needed.

In [ ]:
CHECKPOINT_URL = "https://civitai.com/api/download/models/REPLACE_ME"  # @param {type:"string"}
CIVITAI_TOKEN = ""  # @param {type:"string"}

import os
url = CHECKPOINT_URL
if CIVITAI_TOKEN:
    sep = "&" if "?" in url else "?"
    url = f"{url}{sep}token={CIVITAI_TOKEN}"

os.makedirs("/content/ComfyUI/models/checkpoints", exist_ok=True)
!wget -nc --content-disposition "$url" -P /content/ComfyUI/models/checkpoints
!ls -lh /content/ComfyUI/models/checkpoints

Confirm a `.safetensors` file of several GB shows up above before
continuing. If `ls` only shows a tiny file, the download failed
(usually a bad URL or a missing token) — fix `CHECKPOINT_URL` /
`CIVITAI_TOKEN` and re-run this cell.

## 4. Launch ComfyUI and open a public tunnel to it

In [ ]:
%cd /content/ComfyUI
!wget -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

import subprocess, time, re

comfy_proc = subprocess.Popen(
    ["python3", "main.py", "--listen", "0.0.0.0", "--port", "8188"],
    cwd="/content/ComfyUI",
)
time.sleep(15)  # give ComfyUI time to start before opening the tunnel

tunnel_proc = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://localhost:8188"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print("Waiting for tunnel URL...")
for line in tunnel_proc.stdout:
    print(line, end="")
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        print(f"\n\nOpen ComfyUI here: {match.group(0)}\n")
        break

Click the printed `https://....trycloudflare.com` link — that's your
ComfyUI web UI, same interface as a local install.

**This cell keeps running** (it's what keeps ComfyUI + the tunnel
alive) — leave it running while you work, don't interrupt it until
you're done for the session.

## 5. Generate

In the ComfyUI web UI:
1. Default graph already has `Load Checkpoint` -> prompts ->
   `KSampler` -> `Save Image`. In `Load Checkpoint`, select the file
   you downloaded in step 3.
2. Paste a positive/negative prompt from `docs/art-direction.md`
   (e.g. Faelen's SD/Illustrious draft) into the two `CLIP Text
   Encode` boxes.
3. Use an SDXL-native resolution (1024x1024, or 832x1216 for a
   portrait model sheet) in the `Empty Latent Image` node.
4. Click **Queue Prompt**.

Generated images save to `/content/ComfyUI/output/` on the Colab VM —
download anything you want to keep before the session ends (files
panel on the left, or `!zip` + download the archive).

## 6. (Optional) Download all outputs as a zip

In [ ]:
from google.colab import files
!zip -r /content/gatefall_outputs.zip /content/ComfyUI/output
files.download("/content/gatefall_outputs.zip")